# ReadSigil · Build a Small Language Model

We **fine-tune** the open Qwen2.5 0.5B Instruct model to classify fictional banking-support requests. We do **not** train a foundation model from scratch. No real personal or bank data.

1. Enable **Runtime → Change runtime type → GPU** (if available).
2. Run each cell in order. The notebook downloads an existing model, measures its baseline, trains a LoRA adapter, and measures the same held-out set again.
3. Don't assume an improvement. Read individual errors before deciding whether the adapted model helps.


In [ ]:
!git clone --depth 1 https://github.com/Shashwat-Kumar/shashwat-analytics.git
%cd shashwat-analytics/slm
%pip install -q -r requirements.txt


## Train and compare

The training script uses **75 invented training cases** and **25 separately authored evaluation cases**. It reports baseline and fine-tuned exact-label accuracy without inventing any scores.


In [ ]:
!python train.py --train data/train.jsonl --eval data/eval.jsonl --epochs 3 --rank 8


## Inspect the evidence

A label can be **invalid**, even if some cases look correct. Inspect the mistakes and compare against a rules-based classifier before any serious deployment discussion.


In [ ]:
import json
from pathlib import Path
print(Path('out/metrics.json').read_text())
cases=json.loads(Path('out/individual_predictions.json').read_text())
for stage in ('base','adapted'):
    wrong=[r for r in cases[stage] if not r['correct']]
    print(f'\\n{stage}: {len(wrong)} incorrect or invalid:')
    for row in wrong[:10]:
        print(row)


## Before shipping

The dataset is deliberately small and synthetic. Expect unstable estimates. Do not reuse results as production benchmarks. Define a representative held-out set with consent and data governance, compare with no-training alternatives, test uncommon cases, and measure actual operational cost.

Useful sources: [PEFT](https://huggingface.co/docs/peft/quicktour), [model card](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct), [training guide](https://huggingface.co/docs/transformers/training).
